In [1]:
import sympy as sm
import sympy.physics.mechanics as me
me.init_vprinting()

In [2]:
M, m, l, k1, k2, c1, g, h, w, d, r = sm.symbols('M, m, l, k1, k2, c1, g, h, w, d, r')
q1, q2 = me.dynamicsymbols('q1 q2')
q1d = me.dynamicsymbols('q1', 1)

In [3]:
N = me.ReferenceFrame('N')
B = N.orientnew('B', 'axis', (q2, N.z))

In [4]:
B.ang_vel_in(N)

In [5]:
O = me.Point('O')
block_point = O.locatenew('block', q1 * N.y)
pendulum_point = block_point.locatenew('pendulum', l * B.y)

In [6]:
O.set_vel(N, 0)
block_point.set_vel(N, q1d * N.y)
pendulum_point.v2pt_theory(block_point, N, B)

In [7]:
I_block = M / 12 * me.inertia(N, h**2 + d**2, w**2 + d**2, w**2 + h**2)
I_pendulum = 2*m*r**2/5*me.inertia(B, 1, 0, 1)

In [8]:
block_body = me.RigidBody('block', block_point, N, M, (I_block, block_point))
pendulum_body = me.RigidBody('pendulum', pendulum_point, B, m, (I_pendulum, pendulum_point))


In [9]:
from sympy.physics.mechanics.pathway import LinearPathway

In [10]:
path = me.LinearPathway(O, block_point)
spring = me.DuffingSpring(k1, k2, path, 0)
damper = me.LinearDamper(c1, path)

In [11]:
loads = spring.to_loads() + damper.to_loads()

In [12]:
bodies = [block_body, pendulum_body]

In [13]:
for body in bodies:
    loads.append(me.Force(body, body.mass * g * N.y))

In [14]:
loads

[Force(point=O, force=(k1*sqrt(q1(t)**2) + k2*(q1(t)**2)**(3/2))*q1(t)/sqrt(q1(t)**2)*N.y),
 Force(point=block, force=(-k1*sqrt(q1(t)**2) - k2*(q1(t)**2)**(3/2))*q1(t)/sqrt(q1(t)**2)*N.y),
 Force(point=O, force=c1*Derivative(q1(t), t)*N.y),
 Force(point=block, force=- c1*Derivative(q1(t), t)*N.y),
 Force(point=block, force=M*g*N.y),
 Force(point=pendulum, force=g*m*N.y)]

In [15]:
L = me.Lagrangian(N, block_body, pendulum_body)
L

In [16]:
LM = me.LagrangesMethod(L, [q1, q2], bodies=bodies, forcelist=loads, frame=N)
sm.simplify(LM.form_lagranges_equations())

⎡                              ⎛                           2     ⎞   ⎛         ↪
⎢-M⋅g + M⋅q₁̈ + c₁⋅q₁̇ - g⋅m - m⋅⎝l⋅sin(q₂)⋅q₂̈ + l⋅cos(q₂)⋅q₂̇  - q₁̈⎠ + ⎝k₁ + k₂⋅ ↪
⎢                                                                              ↪
⎢                 ⎛                   2                          2   ⎞         ↪
⎢               m⋅⎝5⋅g⋅l⋅sin(q₂) + 5⋅l ⋅q₂̈ - 5⋅l⋅sin(q₂)⋅q₁̈ + 2⋅r ⋅q₂̈⎠         ↪
⎢               ──────────────────────────────────────────────────────         ↪
⎣                                         5                                    ↪

↪   2⎞   ⎤
↪ q₁ ⎠⋅q₁⎥
↪        ⎥
↪        ⎥
↪        ⎥
↪        ⎥
↪        ⎦